# PatchCore EfficientNet-B1 Layer 3/5 (`x240`)

This notebook is the canonical training and evaluation workflow for the two-layer EfficientNet-B1 PatchCore experiment using layers 3 and 5.

Run this notebook from top to bottom to reproduce the experiment locally. The notebook prepares the data split, trains the model when needed, evaluates the trained model, and writes outputs into the experiment-local artifact folders.

Artifacts written by this notebook:
- `[experiments/anomaly_detection/patchcore/efficientnet_b1/x240/layer3_5/artifacts/checkpoints](experiments/anomaly_detection/patchcore/efficientnet_b1/x240/layer3_5/artifacts/checkpoints)` for model checkpoints
- `[experiments/anomaly_detection/patchcore/efficientnet_b1/x240/layer3_5/artifacts/plots](experiments/anomaly_detection/patchcore/efficientnet_b1/x240/layer3_5/artifacts/plots)` for saved figures
- `[experiments/anomaly_detection/patchcore/efficientnet_b1/x240/layer3_5/artifacts/results](experiments/anomaly_detection/patchcore/efficientnet_b1/x240/layer3_5/artifacts/results)` for metrics, score files, and CSV outputs


## Imports

This cell imports the Python packages used across training, evaluation, plotting, and artifact export.

In [ ]:
# Core imports
import os
import gc
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

print('Using device:', DEVICE)

## Configuration

This cell resolves the repo root, defines experiment settings, and points all outputs to the local artifact folders.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Configuration
DATA_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl')
IMAGE_SIZE = 240
BATCH_SIZE = 128

TRAIN_NORMAL_N = 40_000
TUNE_NORMAL_N  =  5_000
TUNE_DEFECT_N  =   5_000
TEST_NORMAL_N  =  5_000
TEST_DEFECT_N  =   250  

# PatchCore memory/scoring settings
MEMORY_BANK_MAX_PATCHES = 250000
SCORE_CHUNK = 1024
PATCHCORE_NN_K = 3
TOPK_PATCH_RATIO = 0.03
EFFNET_MID_FEATURE_IDX = 3
EFFNET_DEEP_FEATURE_IDX = 5
PATCH_EMBED_DIM = 128

# Threshold sweep settings
THRESHOLD_PERCENTILE_MIN = 90
THRESHOLD_PERCENTILE_MAX = 99.9
THRESHOLD_PERCENTILE_STEPS = 100
THRESHOLD_GRID_STEPS = 300

# Artifact output
ARTIFACT_DIR = str(PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/efficientnet_b1/x240/layer3_5/artifacts')
CHECKPOINTS_DIR = os.path.join(ARTIFACT_DIR, 'checkpoints')
PLOTS_DIR = os.path.join(ARTIFACT_DIR, 'plots')
RESULTS_DIR = os.path.join(ARTIFACT_DIR, 'results')
MODEL_EXPORT_PATH = os.path.join(CHECKPOINTS_DIR, 'patchcore_model.pt')
METRICS_EXPORT_PATH = os.path.join(RESULTS_DIR, 'evaluation_metrics.json')
SCORES_EXPORT_PATH = os.path.join(RESULTS_DIR, 'scores.npz')
RUN_TRAINING = False
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Artifacts will be saved to:', ARTIFACT_DIR)

print(
    f'batch={BATCH_SIZE},bbank_max={MEMORY_BANK_MAX_PATCHES}, '
    f'chunk={SCORE_CHUNK}, amp={USE_CUDA}, topk_ratio={TOPK_PATCH_RATIO}, '
    f'nn_k={PATCHCORE_NN_K}, eff_mid={EFFNET_MID_FEATURE_IDX}, '
    f'eff_deep={EFFNET_DEEP_FEATURE_IDX}, emb_dim={PATCH_EMBED_DIM}'
)
print(
    f'threshold sweep: pct[{THRESHOLD_PERCENTILE_MIN}, {THRESHOLD_PERCENTILE_MAX}] '
    f'steps={THRESHOLD_PERCENTILE_STEPS}, grid_steps={THRESHOLD_GRID_STEPS}'
)


## Load Dataset

This cell loads the legacy WM-811K pickle, cleans labels, and prepares the dataframe used for the experiment split.

In [ ]:
# Load and clean labels
from wafer_defect.data.legacy_pickle import read_legacy_pickle

df = read_legacy_pickle(DATA_PATH)
print('Raw shape:', df.shape)

def parse_failure_label(value):
    if value is None:
        return 'unknown'
    if isinstance(value, float) and np.isnan(value):
        return 'unknown'
    if isinstance(value, (list, tuple, np.ndarray)):
        arr = np.array(value).reshape(-1)
        if len(arr) == 0:
            return 'unknown'
        return str(arr[0])
    return str(value)

df = df.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).astype(str).str.strip()
invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)

normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()

print('Labeled shape:', df.shape)
print('Normal wafers:', len(normal_df))
print('Defect wafers:', len(defect_df))


## Create Split

This cell builds the exact train, tuning, and test split used for this experiment protocol.

In [ ]:
# Enforce requested split sizes with tune split
required_normals = TRAIN_NORMAL_N + TUNE_NORMAL_N + TEST_NORMAL_N
required_defects = TUNE_DEFECT_N + TEST_DEFECT_N

if len(normal_df) < required_normals:
    raise ValueError(f'Not enough normal wafers: need {required_normals}, found {len(normal_df)}')
if len(defect_df) < required_defects:
    raise ValueError(f'Not enough defect wafers: need {required_defects}, found {len(defect_df)}')

rng = np.random.default_rng(SEED)
normal_idx = rng.permutation(len(normal_df))
defect_idx = rng.permutation(len(defect_df))

normal_df_shuf = normal_df.iloc[normal_idx].reset_index(drop=True)
defect_df_shuf = defect_df.iloc[defect_idx].reset_index(drop=True)

n0 = 0
n1 = TRAIN_NORMAL_N
n2 = TRAIN_NORMAL_N + TUNE_NORMAL_N
n3 = TRAIN_NORMAL_N + TUNE_NORMAL_N + TEST_NORMAL_N

train_normal_df = normal_df_shuf.iloc[n0:n1].copy()
tune_normal_df = normal_df_shuf.iloc[n1:n2].copy()
test_normal_df = normal_df_shuf.iloc[n2:n3].copy()

d0 = 0
d1 = TUNE_DEFECT_N
d2 = TUNE_DEFECT_N + TEST_DEFECT_N

tune_defect_df = defect_df_shuf.iloc[d0:d1].copy()
test_defect_df = defect_df_shuf.iloc[d1:d2].copy()

print('Train normal:', len(train_normal_df))
print('Tune normal:', len(tune_normal_df))
print('Tune defect:', len(tune_defect_df))
print('Test normal:', len(test_normal_df))
print('Test defect:', len(test_defect_df))


## Preprocess Images

This cell converts wafer maps into normalized tensor images for the backbone model.

In [ ]:
# Wafer map -> 3ch tensor
def wafer_to_tensor(wafer_map, size=224):
    arr = np.array(wafer_map, dtype=np.int64)
    arr = np.clip(arr, 0, 2)
    x = torch.tensor(arr, dtype=torch.long)
    x = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()
    x = x.unsqueeze(0)
    x = F.interpolate(x, size=(size, size), mode='nearest')
    return x.squeeze(0)

def build_tensor(frame, size=224):
    imgs = [wafer_to_tensor(w, size=size) for w in frame['waferMap'].values]
    X = torch.stack(imgs)
    y = torch.tensor(frame['is_anomaly'].values, dtype=torch.long)
    return X, y

X_train, y_train = build_tensor(train_normal_df, IMAGE_SIZE)
X_tune_normal, y_tune_normal = build_tensor(tune_normal_df, IMAGE_SIZE)
X_tune_defect, y_tune_defect = build_tensor(tune_defect_df, IMAGE_SIZE)
X_test_normal, y_test_normal = build_tensor(test_normal_df, IMAGE_SIZE)
X_test_defect, y_test_defect = build_tensor(test_defect_df, IMAGE_SIZE)

print('X_train:', tuple(X_train.shape))
print('X_tune_normal:', tuple(X_tune_normal.shape))
print('X_tune_defect:', tuple(X_tune_defect.shape))
print('X_test_normal:', tuple(X_test_normal.shape))
print('X_test_defect:', tuple(X_test_defect.shape))


## Build DataLoaders

This cell wraps the prepared datasets into DataLoaders for feature extraction and evaluation.

In [ ]:
# DataLoaders
loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'shuffle': False,
}

train_loader = DataLoader(TensorDataset(X_train, y_train), **loader_kwargs)
tune_normal_loader = DataLoader(TensorDataset(X_tune_normal, y_tune_normal), **loader_kwargs)
tune_defect_loader = DataLoader(TensorDataset(X_tune_defect, y_tune_defect), **loader_kwargs)
test_normal_loader = DataLoader(TensorDataset(X_test_normal, y_test_normal), **loader_kwargs)
test_defect_loader = DataLoader(TensorDataset(X_test_defect, y_test_defect), **loader_kwargs)

print('Train batches:', len(train_loader))
print('Tune normal batches:', len(tune_normal_loader))
print('Tune defect batches:', len(tune_defect_loader))
print('Test normal batches:', len(test_normal_loader))
print('Test defect batches:', len(test_defect_loader))


## Define Model

This cell defines the PatchCore feature extractor and supporting model components for this branch.

In [ ]:
# PatchCore feature extractor from pretrained EfficientNet-B1
class PatchFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = efficientnet_b1(weights=EfficientNet_B1_Weights.DEFAULT)
        self.features = backbone.features
        self.mid_idx = EFFNET_MID_FEATURE_IDX
        self.deep_idx = EFFNET_DEEP_FEATURE_IDX
        self.project_dim = PATCH_EMBED_DIM
        with torch.inference_mode():
            dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
            x = dummy
            f_mid = None
            f_deep = None
            for i, block in enumerate(self.features):
                x = block(x)
                if i == self.mid_idx:
                    f_mid = x
                if i == self.deep_idx:
                    f_deep = x
        if f_mid is None or f_deep is None:
            raise ValueError(f'Invalid EfficientNet feature indices: mid={self.mid_idx}, deep={self.deep_idx}')
        in_dim = f_mid.shape[1] + f_deep.shape[1]
        self.proj = nn.Linear(in_dim, self.project_dim, bias=False)

    def forward(self, x):
        f_mid = None
        f_deep = None
        for i, block in enumerate(self.features):
            x = block(x)
            if i == self.mid_idx:
                f_mid = x
            if i == self.deep_idx:
                f_deep = x
        if f_mid is None or f_deep is None:
            raise RuntimeError(f'Failed to collect EfficientNet feature maps at indices {self.mid_idx} and {self.deep_idx}.')
        return f_mid, f_deep

extractor = PatchFeatureExtractor().to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False

def patch_embeddings(xb):
    with torch.inference_mode():
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
            f2, f3 = extractor(xb)
            f3_up = F.interpolate(f3, size=f2.shape[-2:], mode='bilinear', align_corners=False)
            emb = torch.cat([f2, f3_up], dim=1)
            emb = emb.permute(0, 2, 3, 1).reshape(-1, emb.size(1))
            emb = extractor.proj(emb)
        emb = F.normalize(emb.float(), p=2, dim=1)
    return emb


## Train and Score

This cell builds the PatchCore memory bank, computes anomaly scores, and saves the score bundle needed for later evaluation.

In [ ]:
if RUN_TRAINING or not (os.path.exists(MODEL_EXPORT_PATH) and os.path.exists(SCORES_EXPORT_PATH) and os.path.exists(METRICS_EXPORT_PATH)):
    # Build memory bank from train-normal patches ??keep tensors on GPU to avoid PCIe round-trips
    sampled_patches = []
    estimated_total_patches = None
    total_seen_patches = 0
    sample_ratio = 1.0

    with torch.inference_mode():
        for xb, _ in train_loader:
            xb = xb.to(DEVICE)
            emb = patch_embeddings(xb)           # stays on GPU (no .cpu())
            total_seen_patches += len(emb)

            if estimated_total_patches is None:
                patches_per_image = len(emb) // len(xb)
                estimated_total_patches = patches_per_image * len(train_normal_df)
                sample_ratio = min(1.0, MEMORY_BANK_MAX_PATCHES / estimated_total_patches)
                print('Estimated raw memory bank patches:', estimated_total_patches)
                print('Sampling ratio:', round(sample_ratio, 6))

            if sample_ratio < 1.0:
                keep_n = max(1, int(round(len(emb) * sample_ratio)))
                keep_idx = torch.randperm(len(emb), device=DEVICE)[:keep_n]  # randperm on GPU
                emb = emb[keep_idx]

            sampled_patches.append(emb)          # GPU tensor

    memory_bank = torch.cat(sampled_patches, dim=0)    # already on GPU
    print('Sampled memory bank patches before trim:', len(memory_bank))
    print('Observed raw patches during pass:', total_seen_patches)

    if len(memory_bank) > MEMORY_BANK_MAX_PATCHES:
        keep_idx = torch.randperm(len(memory_bank), device=DEVICE)[:MEMORY_BANK_MAX_PATCHES]
        memory_bank = memory_bank[keep_idx]

    memory_bank = F.normalize(memory_bank, p=2, dim=1).contiguous()   # no .to(DEVICE) needed
    memory_bank_t = memory_bank.t().contiguous()
    print('Final memory bank patches:', len(memory_bank), '| Emb dim:', memory_bank.shape[1])
    print('Memory bank device:', memory_bank.device)
else:
    print(f'Skipping memory-bank construction and reusing saved artifacts from {ARTIFACT_DIR}')


## Reload Scores

This cell reloads the saved score bundle so threshold selection and downstream evaluation use the persisted results.

In [ ]:
if RUN_TRAINING or not (os.path.exists(MODEL_EXPORT_PATH) and os.path.exists(SCORES_EXPORT_PATH) and os.path.exists(METRICS_EXPORT_PATH)):
    # PatchCore scoring with robust aggregation
    # - Patch score: average distance to k nearest memory-bank neighbors
    # - Image score: mean of top-k% most anomalous patch scores

    def min_dist_to_bank(patches, bank_t, chunk=1024, nn_k=3):
        # Embeddings are L2-normalized, so Euclidean distance is derived from cosine similarity.
        mins = []
        for i in range(0, len(patches), chunk):
            p = patches[i:i + chunk]
            sim = p @ bank_t
            k = min(nn_k, sim.shape[1])
            best_sim = sim.topk(k=k, dim=1).values
            dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best_sim, min=0.0))
            mins.append(dist.mean(dim=1))
        return torch.cat(mins, dim=0)

    def score_loader(loader, bank_t, topk_patch_ratio=0.02, nn_k=3):
        img_scores = []
        with torch.inference_mode():
            for xb, _ in loader:
                xb = xb.to(DEVICE)
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
                    f2, f3 = extractor(xb)
                    f3_up = F.interpolate(f3, size=f2.shape[-2:], mode='bilinear', align_corners=False)
                    emb = torch.cat([f2, f3_up], dim=1)
                    emb = emb.permute(0, 2, 3, 1)
                    b, h, w, c = emb.shape
                    emb = emb.reshape(-1, c)
                    emb = extractor.proj(emb)
                emb = F.normalize(emb.float(), p=2, dim=1)

                patch_scores = min_dist_to_bank(emb, bank_t, chunk=SCORE_CHUNK, nn_k=nn_k)
                patch_scores = patch_scores.reshape(b, h * w)
                topk_patches = max(1, int(round(patch_scores.shape[1] * topk_patch_ratio)))
                topk_patches = min(topk_patches, patch_scores.shape[1])
                score = patch_scores.topk(k=topk_patches, dim=1).values.mean(dim=1)
                img_scores.append(score.cpu())

        return torch.cat(img_scores).numpy()

    train_scores = score_loader(
        train_loader,
        memory_bank_t,
        topk_patch_ratio=TOPK_PATCH_RATIO,
        nn_k=PATCHCORE_NN_K,
    )
    tune_normal_scores = score_loader(
        tune_normal_loader,
        memory_bank_t,
        topk_patch_ratio=TOPK_PATCH_RATIO,
        nn_k=PATCHCORE_NN_K,
    )
    tune_defect_scores = score_loader(
        tune_defect_loader,
        memory_bank_t,
        topk_patch_ratio=TOPK_PATCH_RATIO,
        nn_k=PATCHCORE_NN_K,
    )
    test_normal_scores = score_loader(
        test_normal_loader,
        memory_bank_t,
        topk_patch_ratio=TOPK_PATCH_RATIO,
        nn_k=PATCHCORE_NN_K,
    )
    test_defect_scores = score_loader(
        test_defect_loader,
        memory_bank_t,
        topk_patch_ratio=TOPK_PATCH_RATIO,
        nn_k=PATCHCORE_NN_K,
    )

    # Standardize scores using train-normal distribution for stable thresholding.
    train_score_mu = float(np.mean(train_scores))
    train_score_std = float(np.std(train_scores) + 1e-8)

    train_scores_z = (train_scores - train_score_mu) / train_score_std
    tune_normal_scores_z = (tune_normal_scores - train_score_mu) / train_score_std
    tune_defect_scores_z = (tune_defect_scores - train_score_mu) / train_score_std
    test_normal_scores_z = (test_normal_scores - train_score_mu) / train_score_std
    test_defect_scores_z = (test_defect_scores - train_score_mu) / train_score_std
    #save results for threshold tuning and final evaluation
    np.savez_compressed(
        os.path.join(RESULTS_DIR, 'scores.npz'),
        train_scores_z=train_scores_z,
        tune_normal_scores_z=tune_normal_scores_z,
        tune_defect_scores_z=tune_defect_scores_z,
        test_normal_scores_z=test_normal_scores_z,
        test_defect_scores_z=test_defect_scores_z,
    )

    print(f'PatchCore scoring config -> nn_k={PATCHCORE_NN_K}, topk_patch_ratio={TOPK_PATCH_RATIO:.4f}')
    print(f'Train normal score normalization -> mu={train_score_mu:.6f}, std={train_score_std:.6f}')
else:
    print(f'Skipping score generation and reusing saved artifacts from {ARTIFACT_DIR}')
    saved_metrics = json.loads(Path(METRICS_EXPORT_PATH).read_text(encoding='utf-8'))
    with np.load(SCORES_EXPORT_PATH) as data:
        for key in data.files:
            globals()[key] = data[key]
    train_score_mu = float(saved_metrics.get('train_score_mu', 0.0))
    train_score_std = float(saved_metrics.get('train_score_std', 1.0))
    threshold_z = float(saved_metrics.get('threshold_z', 0.0))
    threshold_raw = float(saved_metrics.get('threshold_raw', train_score_mu + threshold_z * train_score_std))


## Reload Scores

This cell reloads the saved score bundle so threshold selection and downstream evaluation use the persisted results.

In [ ]:
#get from save results for threshold tuning and final evaluation
with np.load(os.path.join(RESULTS_DIR, 'scores.npz')) as data:
    train_scores_z = data['train_scores_z']
    tune_normal_scores_z = data['tune_normal_scores_z']
    tune_defect_scores_z = data['tune_defect_scores_z']
    test_normal_scores_z = data['test_normal_scores_z']
    test_defect_scores_z = data['test_defect_scores_z']

# Tune threshold on the labeled tune split (not on test)
y_tune = np.concatenate([
    np.zeros(len(tune_normal_scores_z), dtype=int),
    np.ones(len(tune_defect_scores_z), dtype=int),
])
score_tune = np.concatenate([tune_normal_scores_z, tune_defect_scores_z])

candidate_thresholds = np.unique(
    np.concatenate([
        np.percentile(train_scores_z, np.linspace(THRESHOLD_PERCENTILE_MIN, THRESHOLD_PERCENTILE_MAX, THRESHOLD_PERCENTILE_STEPS)),
        np.linspace(score_tune.min(), score_tune.max(), THRESHOLD_GRID_STEPS),
    ])
)

rows = []
for candidate in candidate_thresholds:
    pred = (score_tune > candidate).astype(int)
    tp = int(((pred == 1) & (y_tune == 1)).sum())
    tn = int(((pred == 0) & (y_tune == 0)).sum())
    fp = int(((pred == 1) & (y_tune == 0)).sum())
    fn = int(((pred == 0) & (y_tune == 1)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    balanced_accuracy = 0.5 * (recall + specificity)

    rows.append({
        'threshold_z': float(candidate),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'specificity': specificity,
        'balanced_accuracy': balanced_accuracy,
        'fp': fp,
        'fn': fn,
    })

threshold_tuning = pd.DataFrame(rows).sort_values(
    ['balanced_accuracy', 'f1', 'precision', 'threshold_z'],
    ascending=[False, False, False, True],
).reset_index(drop=True)

best_row = threshold_tuning.iloc[0]
threshold_z = float(best_row['threshold_z'])
threshold_raw = train_score_mu + threshold_z * train_score_std

print(f"Tuned z-threshold: {threshold_z:.6f} (raw score threshold: {threshold_raw:.6f})")
print(
    f"Precision={best_row['precision']:.4f} | Recall={best_row['recall']:.4f} | "
    f"F1={best_row['f1']:.4f} | Balanced Acc={best_row['balanced_accuracy']:.4f}"
)
display(threshold_tuning.head(10))

plt.figure(figsize=(8, 4))
plt.plot(threshold_tuning['threshold_z'], threshold_tuning['f1'], label='F1')
plt.plot(threshold_tuning['threshold_z'], threshold_tuning['balanced_accuracy'], label='Balanced accuracy')
plt.axvline(threshold_z, color='red', linestyle='--', label='Selected z-threshold')
plt.xlabel('Z-threshold')
plt.ylabel('Metric')
plt.title('Threshold Tuning Sweep (Tune Split)')
plt.legend()
plt.tight_layout()
plt.show()

print('Run the next evaluation cell to report final test metrics with the tuned threshold.')


## Evaluate

This cell computes the final evaluation metrics on the held-out split using the selected decision threshold.

In [ ]:
# threshold_z = 1.2
# Final evaluation on test split (kept untouched during threshold tuning)
y_true = np.concatenate([
    np.zeros(len(test_normal_scores_z), dtype=int),
    np.ones(len(test_defect_scores_z), dtype=int),
])
scores = np.concatenate([test_normal_scores_z, test_defect_scores_z])
y_pred = (scores > threshold_z).astype(int)

roc_auc = float(roc_auc_score(y_true, scores))
report = classification_report(y_true, y_pred, target_names=['normal', 'anomaly'])

print(f'ROC-AUC (z-score): {roc_auc:.4f}')
print(f'Applied z-threshold: {threshold_z:.6f} | raw-threshold: {threshold_raw:.6f}')
print(report)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4.6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['pred normal', 'pred anomaly'],
            yticklabels=['true normal', 'true anomaly'])
plt.title('PatchCore Confusion Matrix (Test Split)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.kdeplot(test_normal_scores_z, label='Normal test', fill=True, alpha=0.35)
sns.kdeplot(test_defect_scores_z, label='Defect test', fill=True, alpha=0.35)
plt.axvline(threshold_z, color='red', linestyle='--', label=f'Z-threshold {threshold_z:.3f}')
plt.xlabel('PatchCore anomaly score (z-score)')
plt.ylabel('Density')
plt.title('PatchCore Score Distribution (Test Split)')
plt.legend()
plt.tight_layout()
plt.show()
print('test defect samples:', TEST_DEFECT_N)

# Store model artifacts
if RUN_TRAINING or not os.path.exists(MODEL_EXPORT_PATH):
    artifact = {
        'extractor_state_dict': extractor.state_dict(),
        # 'memory_bank': memory_bank.detach().cpu(),
        'threshold_z': float(threshold_z),
        'threshold_raw': float(threshold_raw),
        'train_score_mu': float(train_score_mu),
        'train_score_std': float(train_score_std),
        'config': {
            'image_size': IMAGE_SIZE,
            'train_normal_n': TRAIN_NORMAL_N,
            'tune_normal_n': TUNE_NORMAL_N,
            'tune_defect_n': TUNE_DEFECT_N,
            'test_normal_n': TEST_NORMAL_N,
            'test_defect_n': TEST_DEFECT_N,
            'score_chunk': SCORE_CHUNK,
            'patchcore_nn_k': PATCHCORE_NN_K,
            'patchcore_topk_patch_ratio': TOPK_PATCH_RATIO,
            'effnet_mid_feature_idx': EFFNET_MID_FEATURE_IDX,
            'effnet_deep_feature_idx': EFFNET_DEEP_FEATURE_IDX,
            'patch_embed_dim': PATCH_EMBED_DIM,
        },
    }
    torch.save(artifact, MODEL_EXPORT_PATH)
    else:
    artifact = torch.load(MODEL_EXPORT_PATH, map_location='cpu')
    print('Reusing saved model artifact from:', MODEL_EXPORT_PATH)

metrics = {
    'roc_auc_z': roc_auc,
    'threshold_z': float(threshold_z),
    'threshold_raw': float(threshold_raw),
    'train_score_mu': float(train_score_mu),
    'train_score_std': float(train_score_std),
    'confusion_matrix': cm.tolist(),
    'n_test_normal': int(len(test_normal_scores_z)),
    'n_test_defect': int(len(test_defect_scores_z)),
}
pd.Series(metrics).to_json(METRICS_EXPORT_PATH, indent=2)

print('Saved metrics to:', METRICS_EXPORT_PATH)


## Failure Analysis

This cell summarizes defect-level behavior and writes per-sample analysis outputs for qualitative review.

In [ ]:
#print out each defected wafer's score and label for manual inspection
# Per-class breakdown
tmp = test_defect_df.copy()
tmp['score']    = test_defect_scores_z
tmp['detected'] = (test_defect_scores_z > threshold_z).astype(int)
print('\nPer-defect-class recall:')
display(tmp.groupby('failure_label').agg(
    count=('detected','count'), detected=('detected','sum'),
    recall=('detected','mean'), mean_score=('score','mean'),
).round(3).sort_values('recall'))
    


## Train and Score

This cell builds the PatchCore memory bank, computes anomaly scores, and saves the score bundle needed for later evaluation.

In [ ]:
# Clear memory after run
vars_to_clear = [
    'train_dataset', 'test_normal_dataset', 'test_defect_dataset',
    'sample_x', 'sample_y',
    'sampled_patches', 'memory_bank', 'memory_bank_t',
    'train_scores', 'tune_normal_scores', 'tune_defect_scores',
    'test_normal_scores', 'test_defect_scores',
    'train_scores_z', 'tune_normal_scores_z', 'tune_defect_scores_z',
    'test_normal_scores_z', 'test_defect_scores_z',
    'scores', 'y_true', 'y_pred',
    'train_loader', 'tune_normal_loader', 'tune_defect_loader',
    'test_normal_loader', 'test_defect_loader',
]
for name in vars_to_clear:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print('Memory cleared (Python GC + CUDA cache).')


## Reload Scores

This cell reloads the saved score bundle so threshold selection and downstream evaluation use the persisted results.

In [ ]:
artifact_root = REPO_ROOT / "experiments/anomaly_detection/patchcore/efficientnet_b1/x240/layer3_5" / "artifacts"
checkpoints_dir = artifact_root / "checkpoints"
plots_dir = artifact_root / "plots"
results_dir = artifact_root / "results"
checkpoints_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

import torch

checkpoint_candidates = sorted(checkpoints_dir.glob("*.pt"))
if not checkpoint_candidates:
    raise FileNotFoundError(f"No checkpoint found under {checkpoints_dir}")

checkpoint_path = checkpoint_candidates[0]
checkpoint = torch.load(checkpoint_path, map_location="cpu")
if isinstance(checkpoint, dict):
    print(f"Loaded checkpoint: {checkpoint_path.name} with keys: {list(checkpoint.keys())[:8]}")
else:
    print(f"Loaded checkpoint: {checkpoint_path.name}")

scores_path = results_dir / "scores.npz"
metrics_path = results_dir / "evaluation_metrics.json"
if not scores_path.exists():
    raise FileNotFoundError(f"Saved score bundle not found: {scores_path}")
if not metrics_path.exists():
    raise FileNotFoundError(f"Saved evaluation metrics not found: {metrics_path}")

scores = np.load(scores_path)
evaluation_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
threshold_z = float(evaluation_metrics["threshold_z"])
confusion_matrix = np.array(evaluation_metrics["confusion_matrix"], dtype=int)

series = []
for key, label, color in [
    ("train_scores_z", "train", "#8d99ae"),
    ("tune_normal_scores_z", "val normal", "#457b9d"),
    ("tune_defect_scores_z", "val defect", "#f4a261"),
    ("test_normal_scores_z", "test normal", "#577590"),
    ("test_defect_scores_z", "test defect", "#e76f51"),
]:
    if key in scores:
        series.append((label, np.asarray(scores[key]).astype(float), color))

if not series:
    raise ValueError("No score arrays found in scores.npz")

fig, ax = plt.subplots(figsize=(9, 4.8))
for label, values, color in series:
    ax.hist(values, bins=40, alpha=0.45, label=label, color=color, density=True)
ax.axvline(threshold_z, color="black", linestyle="--", label=f"threshold z={threshold_z:.3f}")
ax.set_title("Score Distribution by Split")
ax.set_xlabel("z-scored wafer anomaly score")
ax.set_ylabel("density")
ax.legend()
score_distribution_path = plots_dir / "score_distribution.png"
plt.tight_layout()
plt.savefig(score_distribution_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved score distribution plot to {score_distribution_path}")

fig, ax = plt.subplots(figsize=(8.5, 4.8))
labels = [label for label, _, _ in series]
values = [vals for _, vals, _ in series]
ax.boxplot(values, labels=labels, showfliers=False)
ax.axhline(threshold_z, color="black", linestyle="--", label="threshold")
ax.set_title("Score Summary by Split")
ax.set_ylabel("z-scored wafer anomaly score")
ax.tick_params(axis="x", rotation=20)
ax.legend()
score_summary_path = plots_dir / "score_summary.png"
plt.tight_layout()
plt.savefig(score_summary_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved score summary plot to {score_summary_path}")

fig, ax = plt.subplots(figsize=(4.8, 4.2))
im = ax.imshow(confusion_matrix, cmap="Blues")
for (row_idx, col_idx), value in np.ndenumerate(confusion_matrix):
    ax.text(col_idx, row_idx, f"{value}", ha="center", va="center", color="black")
ax.set_xticks([0, 1], ["pred normal", "pred anomaly"])
ax.set_yticks([0, 1], ["true normal", "true anomaly"])
ax.set_title("Confusion Matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
confusion_matrix_path = plots_dir / "confusion_matrix.png"
plt.tight_layout()
plt.savefig(confusion_matrix_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved confusion matrix plot to {confusion_matrix_path}")
